# FlashRank-Pro — BEIR Fine-Tune (Bulletproof + Resumable)

**What this does:** Fine-tunes `eulogik/flashrank-pro-beir-step400` on scifact+fiqa+nfcorpus train splits with BM25 hard negatives (margin+BCE loss — identical to `training/05_beir_finetune.py`), then evaluates on full scifact test.

**Bulletproof rules:**
1. **Every stateful artifact lives on Google Drive** (`MyDrive/flashrank_pro/`): BEIR data, training examples, checkpoints, final model. A dead session loses nothing.
2. **Full checkpoint (weights+optimizer+scheduler+step) saved every 100 steps** to Drive. Session dies → reopen → Run all → auto-resumes (~seconds).
3. **All cells are idempotent** — running twice never corrupts state.
4. **fp16 autocast + GradScaler** (T4 has no bf16).

**Run:** Runtime → Run all. Expected on free T4: data build ~20 min (once, cached) → training ~1.5h → eval ~2 min.

Known minor: on resume, the in-progress epoch restarts from its beginning (duplicates ≤200 steps).

In [ ]:
!pip install -q beir rank_bm25 huggingface_hub
import os, sys, json, time, shutil, subprocess, pickle, random, re, traceback
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from rank_bm25 import BM25Okapi

from huggingface_hub import login
try:
    from google.colab import userdata
    login(userdata.get('HF_TOKEN'))
    print('HF auth: OK (Colab secret HF_TOKEN)')
except Exception as _e:
    print('!! HF auth FAILED:', _e)
    print('   -> add a Colab secret named HF_TOKEN (key icon, left panel), then rerun')

torch.set_num_threads(4)
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0), '|', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
else:
    print('WARNING: no GPU — training will be very slow')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/flashrank_pro'
for sub in ('data', 'examples', 'ckpt', 'final'):
    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)
print('Drive ready at', DRIVE)

In [ ]:
DATASETS = ['nfcorpus', 'scifact', 'fiqa', 'webis-touche2020', 'arguana', 'scidocs']
BASE_URL = 'https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/'

def ensure_beir(name):
    d = f'{DRIVE}/data/beir/{name}'
    if os.path.exists(f'{d}/corpus.jsonl'):
        return d
    z = f'{DRIVE}/data/{name}.zip'
    if not os.path.exists(z):
        print(f'  downloading {name}...')
        subprocess.run(['wget', '-q', '-O', z, f'{BASE_URL}{name}.zip'], check=True)
    print(f'  extracting {name}...')
    subprocess.run(['unzip', '-o', '-q', z, '-d', f'{DRIVE}/data/beir/'], check=True)
    if not os.path.exists(f'{d}/corpus.jsonl'):
        print(f'  WARNING: {d} missing after extract; listing:', os.listdir(f'{DRIVE}/data/beir/'))
        raise RuntimeError(f'extract failed for {name}')
    return d

for n in DATASETS + ['scifact']:
    ensure_beir(n)
print('BEIR data ready:', [n for n in DATASETS])

In [ ]:
def tokenize(text: str) -> list:
    return re.findall(r"[a-z0-9]+", text.lower())

def doc_text(x):
    return x['text'] if isinstance(x, dict) else str(x)

def query_text(x):
    return x if isinstance(x, str) else x.get('text', '')

def rel_score(v):
    if isinstance(v, dict):
        v = v.get('score', v.get('relevance', 1))
    return int(v)

def norm_qrels(qrels):
    return {q: {d: rel_score(s) for d, s in rel.items()} for q, rel in qrels.items()}

def load_split(name, split):
    from beir.datasets.data_loader import GenericDataLoader
    return GenericDataLoader(data_folder=f'{DRIVE}/data/beir/{name}').load(split=split)

def build_index(name, corpus):
    ids = list(corpus.keys())
    bm25 = BM25Okapi([tokenize(doc_text(corpus[i])) for i in ids], k1=1.5, b=0.75)
    return bm25, ids

def build_examples(datasets, max_q, n_neg, seed=0):
    random.seed(seed)
    examples = []
    for name in datasets:
        t0 = time.time()
        try:
            corpus, queries, qrels = load_split(name, 'train')
        except Exception as e:
            print(f'  {name}: no train split ({e}); skipping'); continue
        qrels = norm_qrels(qrels)
        bm25, doc_ids = build_index(name, corpus)
        pos_map = {qid: [d for d, s in qrels[qid].items() if s > 0] for qid in qrels}
        qids = [q for q in queries if q in pos_map]
        random.shuffle(qids)
        qids = qids[:max_q]
        n = 0
        for qid in qids:
            pos = [doc_text(corpus[d]) for d in pos_map[qid] if d in corpus]
            if not pos:
                continue
            scores = bm25.get_scores(tokenize(query_text(queries[qid])))
            order = np.argsort(scores)[::-1]
            negs = [doc_text(corpus[doc_ids[i]]) for i in order if doc_ids[i] not in pos_map[qid]][:n_neg]
            if len(negs) < 2:
                continue
            examples.append({'query': query_text(queries[qid]), 'pos': pos[:2], 'negs': negs})
            n += 1
        print(f'  {name}: {n} examples ({time.time()-t0:.0f}s)')
    return examples

MSMARCO_EX = f'{DRIVE}/examples/msmarco_train.jsonl'
EXAMPLES = None
if os.path.exists(MSMARCO_EX):
    try:
        with open(MSMARCO_EX) as f:
            EXAMPLES = [json.loads(line) for line in f if line.strip()]
        print(f'loaded {len(EXAMPLES)} MS MARCO examples from Drive')
    except json.JSONDecodeError as e:
        print(f'WARNING: {MSMARCO_EX} is corrupted ({e}) — deleting and rebuilding')
        os.remove(MSMARCO_EX)
        EXAMPLES = None
if EXAMPLES is None:
    EX_PKL = f'{DRIVE}/examples/train_examples.pkl'
    if os.path.exists(EX_PKL):
        with open(EX_PKL, 'rb') as f:
            EXAMPLES = pickle.load(f)
        if len(EXAMPLES) == 0:
            print('cached examples are EMPTY (previous failed run) — rebuilding')
            EXAMPLES = None
        else:
            print(f'loaded {len(EXAMPLES)} cached examples')
    else:
        EXAMPLES = None
    if EXAMPLES is None:
        print('building training examples (once, ~20 min)...')
        EXAMPLES = build_examples(DATASETS, 4000, 5, seed=0)
        with open(EX_PKL, 'wb') as f:
            pickle.dump(EXAMPLES, f)
        print(f'saved {len(EXAMPLES)} examples to Drive')
assert len(EXAMPLES) > 0, 'no training examples built — check the BEIR data cell above'
print('sample:', EXAMPLES[0]['query'][:60], '| pos', len(EXAMPLES[0]['pos']), '| negs', len(EXAMPLES[0]['negs']))

In [ ]:
# MANUAL RESET — set RESET = True and run this cell ONCE to discard the previous (buggy-loss)
# training run so the fixed-loss run starts fresh from MODEL_INIT.
# Keep RESET = False during normal use / Run all — it is a no-op and cannot wipe a resumable checkpoint.
RESET = False
if RESET:
    import shutil
    for d in (f'{DRIVE}/ckpt', f'{DRIVE}/final'):
        if os.path.exists(d):
            shutil.rmtree(d)
            print('deleted', d)
    ckpt = None
    model = None
    tokenizer = None
    print('Drive reset complete — next Run all loads a fresh model from MODEL_INIT')
else:
    print('RESET not triggered (RESET = False) — nothing deleted.')

In [ ]:
MODEL_INIT = 'eulogik/flashrank-pro-beir-step400'   # start point (our step-400 checkpoint)
CKPT_DIR = f'{DRIVE}/ckpt'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def load_checkpoint_state():
    p = os.path.join(CKPT_DIR, 'checkpoint.pt')
    if os.path.exists(p):
        return torch.load(p, map_location='cpu')
    return None

ckpt = load_checkpoint_state()
best_dir = os.path.join(CKPT_DIR, 'model')
if ckpt is not None and not os.path.exists(os.path.join(best_dir, 'config.json')):
    print('  !! checkpoint.pt exists but ckpt/model missing -> discarding stale checkpoint')
    ckpt = None
if ckpt is not None:
    model = AutoModelForSequenceClassification.from_pretrained(os.path.join(CKPT_DIR, 'model'))
    print('RESUMING from Drive checkpoint at step', ckpt['step'])
else:
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_INIT, trust_remote_code=True)
    print('Fresh start from', MODEL_INIT)
tokenizer = AutoTokenizer.from_pretrained(os.path.join(CKPT_DIR, 'model')) if ckpt is not None \
    else AutoTokenizer.from_pretrained(MODEL_INIT, trust_remote_code=True)
print('params:', round(sum(p.numel() for p in model.parameters()) / 1e6), 'M')
model.to(DEVICE)
USE_GC = False
USE_AMP = False   # pure fp32: fp16-autocast backward yields scale-independent nan grads on current Colab stack
if USE_GC and hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    print('gradient checkpointing enabled')
else:
    print('gradient checkpointing OFF (GC+fp16-autocast backward produces nan grads on this stack)')

In [ ]:
def evaluate_beir(name, model, tokenizer, top_k=100, max_q=None):
    from beir.retrieval.evaluation import EvaluateRetrieval
    corpus, queries, qrels_raw = load_split(name, 'test')
    qrels = norm_qrels(qrels_raw)
    if max_q:
        qids = list(queries.keys())[:max_q]
        queries = {q: queries[q] for q in qids}
    ids = list(corpus.keys())
    docs_text = [doc_text(corpus[d]) for d in ids]
    bm25 = BM25Okapi([tokenize(t) for t in docs_text], k1=1.5, b=0.75)
    model.eval()
    pairs_all, meta = [], []
    for qid, q in queries.items():
        q_text = query_text(q)
        scores = bm25.get_scores(tokenize(q_text))
        top = np.argsort(scores)[::-1][:top_k]
        for i in top:
            meta.append((qid, ids[i]))
            pairs_all.append(q_text + ' </s> ' + docs_text[i])
    logits = []
    with torch.no_grad():
        for i in range(0, len(pairs_all), 128):
            chunk = pairs_all[i:i+128]
            t = tokenizer(chunk, truncation=True, max_length=512, padding=True, return_tensors='pt')
            logits.append(model(input_ids=t['input_ids'].to(DEVICE),
                                attention_mask=t['attention_mask'].to(DEVICE)).logits.float().squeeze(-1).cpu())
    logits = torch.cat(logits)
    scores = torch.sigmoid(logits)
    reranked = {}
    for (qid, did), s in zip(meta, scores.tolist()):
        reranked.setdefault(qid, {})[did] = float(s)
    metrics = EvaluateRetrieval.evaluate(qrels, reranked, [10])[0]
    return {'NDCG@10': float(metrics['NDCG@10'])}

In [ ]:
class RerankDS(Dataset):
    def __init__(self, ex, tok, max_len):
        self.ex, self.tok, self.max_len = ex, tok, max_len
    def __len__(self):
        return len(self.ex)
    def __getitem__(self, i):
        e = self.ex[i]
        q = e['query']
        docs = e['pos'][:1] + e['negs']
        labels = [1.0] + [0.0] * len(e['negs'])
        ts = self.tok([q + ' </s> ' + d for d in docs], truncation=True, max_length=self.max_len, padding=False)
        return ts['input_ids'], ts['attention_mask'], torch.tensor(labels, dtype=torch.float)

def collate(batch, tok, max_len):
    ids, masks, labels = zip(*batch)
    n_docs = [len(x) for x in ids]
    flat_ids, flat_masks, flat_labels, bi = [], [], [], []
    for qi, (x, m, l) in enumerate(zip(ids, masks, labels)):
        for j in range(len(x)):
            flat_ids.append(x[j]); flat_masks.append(m[j])
            flat_labels.append(l[j]); bi.append(qi)
    L = max(len(x) for x in flat_ids)
    ids = torch.tensor([x + [tok.pad_token_id] * (L - len(x)) for x in flat_ids])
    masks = torch.tensor([m + [0] * (L - len(m)) for m in flat_masks])
    return ids, masks, torch.tensor(flat_labels, dtype=torch.float), torch.tensor(bi), n_docs

def run_training(model, tokenizer, examples, args, resume_state):
    ds = RerankDS(examples, tokenizer, args['max_length'])
    loader = DataLoader(ds, batch_size=args['batch_size'], shuffle=True,
                       collate_fn=lambda b: collate(b, tokenizer, args['max_length']), drop_last=True)
    steps_per_epoch = len(loader) // args['grad_accum']
    total_opt_steps = steps_per_epoch * args['epochs']
    optimizer = torch.optim.AdamW(model.parameters(), lr=args['lr'], weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(optimizer, 100, total_opt_steps)
    scaler = torch.amp.GradScaler('cuda', enabled=(USE_AMP and DEVICE == 'cuda'))

    step = 0; best = 0.0; ema = None; n_skipped = 0
    if resume_state:
        step = resume_state['step']
        best = resume_state.get('best', 0.0)
        optimizer.load_state_dict(resume_state['optimizer'])
        scheduler.load_state_dict(resume_state['scheduler'])
        print(f'  resumed optimizer/scheduler at step {step}')

    epoch0 = step // steps_per_epoch
    start_t = time.time()
    def save_ckpt():
        os.makedirs(CKPT_DIR, exist_ok=True)
        torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(), 'step': step, 'best': best},
                   os.path.join(CKPT_DIR, 'checkpoint.pt'))
        print(f'  [checkpoint.pt -> Drive @ step {step}]', flush=True)
    def save_best():
        mdir = os.path.join(CKPT_DIR, 'model')
        os.makedirs(mdir, exist_ok=True)
        model.save_pretrained(mdir); tokenizer.save_pretrained(mdir)
        print(f'  [best model -> Drive @ step {step}]', flush=True)

    for ep in range(epoch0, args['epochs']):
        ep_opt_steps = (ep + 1) * steps_per_epoch
        for bi, (ids, mask, labels, bi_map, n_docs) in enumerate(loader):
            if step >= ep_opt_steps:
                break
            if USE_AMP:
                with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
                    logits = model(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE)).logits.float().squeeze(-1)
                scores = torch.sigmoid(logits.float())
            else:
                logits = model(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE)).logits.float().squeeze(-1)
                scores = torch.sigmoid(logits)
            loss = torch.zeros((), dtype=torch.float32, device=DEVICE)
            margin_loss = torch.zeros((), dtype=torch.float32, device=DEVICE)
            bce = torch.zeros((), dtype=torch.float32, device=DEVICE)
            n_pos = 0
            for qi in range(len(n_docs)):
                sel = bi_map == qi
                ls = labels[sel]; ss = scores[sel]; lg = logits[sel]
                pos_m = ls == 1.0; neg_m = ls == 0.0
                pos_s = ss[pos_m]; neg_s = ss[neg_m]
                pos_l = lg[pos_m]; neg_l = lg[neg_m]
                if len(pos_s) == 0 or len(neg_s) == 0:
                    continue
                for pl, ps in zip(pos_l, pos_s):
                    margin_loss += torch.clamp(args['margin'] - (ps - neg_s), min=0.0).sum()
                    bce += F.softplus(-pl) + F.softplus(neg_l).mean()
                    n_pos += 1
            margin_loss = margin_loss / max(n_pos, 1)
            bce = bce / max(n_pos, 1)
            loss = margin_loss + args['bce_weight'] * bce
            if USE_AMP:
                scaler.scale(loss).backward()
            else:
                loss.backward()
            if (bi + 1) % args['grad_accum'] == 0:
                if USE_AMP:
                    scaler.unscale_(optimizer)
                gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                if not torch.isfinite(gnorm):
                    n_skipped += 1
                    bad = [n for n, p in model.named_parameters() if p.grad is not None and not bool(torch.isfinite(p.grad).all())]
                    optimizer.zero_grad(set_to_none=True)
                    print(f"  !! NONFINITE GRADS #{n_skipped}, gnorm={float(gnorm):.3f}, {len(bad)}/{sum(1 for p in model.parameters() if p.grad is not None)} params bad, first: {bad[:4]}", flush=True)
                else:
                    if USE_AMP:
                        scaler.step(optimizer); scaler.update()
                    else:
                        optimizer.step()
                    scheduler.step()
                step += 1
                ema = loss.item() if ema is None else 0.95 * ema + 0.05 * loss.item()
                if step % 50 == 0:
                    rate = (time.time() - start_t) / (step - (resume_state['step'] if resume_state else 0) + 1)
                    print(f'  step {step}/{total_opt_steps} | loss {ema:.4f} (margin {margin_loss.item():.4f}) | {rate:.2f}s/step | ETA {(total_opt_steps-step)*rate/60:.0f}min', flush=True)
                if step % 100 == 0:
                    save_ckpt()
                if step % args['eval_every'] == 0:
                    model.eval()
                    with torch.no_grad():
                        m = evaluate_beir('scifact', model, tokenizer, max_q=40)
                    model.train()
                    ndcg = m.get('NDCG@10', 0.0)
                    print(f'  >>> EVAL scifact(40q): NDCG@10 = {ndcg:.4f} (best {best:.4f}) [skipped updates so far: {n_skipped}]', flush=True)
                    if ndcg > best:
                        best = ndcg; save_best()
                    save_ckpt()
    save_ckpt()
    print('TRAINING COMPLETE')
    return model, step, best

ARGS = {'max_length': 256, 'batch_size': 4, 'grad_accum': 4, 'epochs': 2, 'lr': 2e-5, 'margin': 0.15, 'bce_weight': 0.5, 'eval_every': 200}
ckpt = load_checkpoint_state()
best_dir = os.path.join(CKPT_DIR, 'model')
if ckpt is not None and not os.path.exists(os.path.join(best_dir, 'config.json')):
    print('  !! checkpoint.pt exists but ckpt/model missing -> discarding stale checkpoint')
    ckpt = None
if ckpt is not None:
    model = AutoModelForSequenceClassification.from_pretrained(os.path.join(CKPT_DIR, 'model'))
    tokenizer = AutoTokenizer.from_pretrained(os.path.join(CKPT_DIR, 'model'))
    model.to(DEVICE)
else:
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_INIT, trust_remote_code=True)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_INIT, trust_remote_code=True)
    model.to(DEVICE)
    print('Fresh start from', MODEL_INIT)
if USE_GC and hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    print('gradient checkpointing enabled')
total_est = (len(EXAMPLES) // (ARGS['batch_size'] * ARGS['grad_accum'])) * ARGS['epochs']
if ckpt is not None and ckpt['step'] >= total_est:
    print(f'  !!! WARNING: Drive checkpoint is COMPLETE (step {ckpt["step"]} >= est. total {total_est}).')
    print('  !!! Resuming would do NOTHING (zero training steps). To retrain with the fixed loss,')
    print('  !!! set RESET = True in the MANUAL RESET cell above and run it once, then Run all.')
model, step, best = run_training(model, tokenizer, EXAMPLES, ARGS, ckpt)
print(f'FINAL: step {step}, best scifact(40q) NDCG@10 {best:.4f}')

In [ ]:
model.eval()
results = {}
for n in DATASETS:
    try:
        m = evaluate_beir(n, model, tokenizer)
        results[n] = m
        print(f'[{n}] NDCG@10 = {m["NDCG@10"]:.4f}')
    except Exception as e:
        print(f'[{n}] ERROR: {e}')
avg = np.mean([v['NDCG@10'] for v in results.values()])
print(f'\nAVERAGE NDCG@10: {avg:.4f}')
with open(f'{DRIVE}/final/beir_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('saved to', f'{DRIVE}/final/beir_results.json')

In [ ]:
final_dir = f'{DRIVE}/final/model'
os.makedirs(final_dir, exist_ok=True)
model.save_pretrained(final_dir); tokenizer.save_pretrained(final_dir)
print('final model saved to', final_dir)
print('All done. Publish later via deploy_to_huggingface.py.')